In [1]:
import numpy as np
from scipy.integrate import fixed_quad, quad
import os
import plotly.graph_objects as go
from scipy.special import j0, jv
import pandas as pd

In [2]:
# Leitura do arquivo com separação por espaços
data_atlas = pd.read_csv(
    "../../../data/sigma_tot_2/ensemble_StRh_atlas.dat",
    delim_whitespace=True,
    header=None,
    nrows=70  # lê apenas as 70 primeiras linhas
)

x_atlas = data_atlas[0].to_numpy()
y_atlas = data_atlas[1].to_numpy()
y_error_atlas = data_atlas[2].to_numpy()

/tmp/ipykernel_21498/3003301750.py:2: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  data_atlas = pd.read_csv(


In [3]:
# === Global Configuration and Constants ===
start_sqrt_s = 101  # Global parameter controlling energy scale
b_0 = (33 - 6) / (12 * np.pi)  # β0 for nf=3
Lambda = 0.284  # ΛQCD in GeV
gamma_1 = 0.084
gamma_2 = 2.36
rho = 4.0

lst_sigma_tot_born = []
lst_sqrt_s = []
lst_error = []

s0 = 1.0  # GeV^2

epsilon_atlas = 0.0732

max_sqrt_s = 13000
step = 100
n_points = 10000

model_params = {
    'atlas': {
        'pl':  {'mg': 0.417, 'a1': 1.563, 'a2': 2.22}
    }
}

epsilon_values = {
    'atlas': epsilon_atlas
}

lst_amp_born = []
lst_sqrt_s = []
lst_sigma_tot_born = []




In [4]:
# === Auxiliary Functions for Physical Model ===
def m2_pl(q2, mg):
    lambda_squared = Lambda ** 2
    rho_mg_squared = rho * mg ** 2
    ratio = np.log((q2 + rho_mg_squared) / lambda_squared) / np.log(rho_mg_squared / lambda_squared)
    return (mg ** 4 / (q2 + mg ** 2)) * ratio ** (gamma_2 - 1)

def get_m2_function(mass_model):
    return m2_pl

def G_p(q2, a1, a2):
    return np.exp(-(a1 * q2 + a2 * q2 ** 2))

def alpha_D(q2, mg, m2_func):
    m2 = m2_func(q2, mg)
    return 1.0 / (b_0 * (q2 + m2) * np.log((q2 + 4 * m2) / (Lambda ** 2)))

def T_1(k, phi, mg, a1, a2, m2_func, q):
    q2 = q ** 2
    qk_cos = q * k * np.cos(phi)
    qk_plus_squared = q2 / 4 + qk_cos + k ** 2
    qk_minus_squared = q2 / 4 - qk_cos + k ** 2
    alpha_D_plus = alpha_D(qk_plus_squared, mg, m2_func)
    alpha_D_minus = alpha_D(qk_minus_squared, mg, m2_func)
    G0 = G_p(q2, a1, a2)
    return alpha_D_plus * alpha_D_minus * G0 ** 2

def T_2(k, phi, mg, a1, a2, m2_func, q):
    q2 = q ** 2
    qk_cos = q * k * np.cos(phi)
    qk_plus_squared = q2 / 4 + qk_cos + k ** 2
    qk_minus_squared = q2 / 4 - qk_cos + k ** 2
    alpha_D_plus = alpha_D(qk_plus_squared, mg, m2_func)
    alpha_D_minus = alpha_D(qk_minus_squared, mg, m2_func)
    factor = q2 + 9 * abs(k ** 2 - q2 / 4)
    G0 = G_p(q2, a1, a2)
    G_minus = G_p(factor, a1, a2)
    return alpha_D_plus * alpha_D_minus * G_minus * (2 * G0 - G_minus)


# -------------------------------
# Inner integral (over phi)
# -------------------------------
def phi_integral(k, mg, a1, a2, m2_func, q, n_points=10000):
    integrand = lambda phi: k * (T_1(k, phi, mg, a1, a2, m2_func, q) - 
                                 T_2(k, phi, mg, a1, a2, m2_func, q))
    result, _ = fixed_quad(integrand, 0, 2*np.pi, n=n_points)
    return result

# -------------------------------
# Outer integral (over k)
# -------------------------------
def k_integral(k, mg, a1, a2, m2_func, q, n_points=10000):
    return phi_integral(k, mg, a1, a2, m2_func, q, n_points)

# -------------------------------
# Double integral computation
# -------------------------------
def compute_double_integral(sqrt_s_val, mg, a1, a2, m2_func, q, n_points=10000):
    result, _ = fixed_quad(
        lambda k: k_integral(k, mg, a1, a2, m2_func, q, n_points),
        0, sqrt_s_val, 
        n=n_points
    )
    return result

def born_amp(diff_T, s, epsilon, t):
    
    alpha_pomeron = 1.0 + epsilon + 0.25 * t

    first_term = s**(alpha_pomeron)
    second_term = 1/(1**(alpha_pomeron-1))
    
    regge_factor = first_term * second_term
    return 1j * 8.0 * regge_factor * diff_T

def sigma_tot_born(amp_born_value, s):
    return amp_born_value.imag / s * 0.389379323


In [5]:

def calculate_born_cross_sections(start_sqrt_s, max_sqrt_s, step, mg, a1, a2, m2_func, epsilon, n_points=10000):
    """Calculate cross sections for all sqrt_s values"""
    
    # Generate array of sqrt_s values
    sqrt_s_values = np.arange(start_sqrt_s, max_sqrt_s + step, step)
    
    # Process each sqrt_s value
    lst_sigma_tot_born = []
    lst_sqrt_s = []
    lst_amp_born = []
    
    for sqrt_s_val in sqrt_s_values:
        # Compute the double integral
        q = 0  
        diff_T = compute_double_integral(sqrt_s_val, mg, a1, a2, m2_func, q, n_points)
        
        # Calculate amplitude and cross section
        s = sqrt_s_val * sqrt_s_val
        amp_born_value = born_amp(diff_T, s, epsilon, 0)
        sigma_tot_born_value = sigma_tot_born(amp_born_value, s)
        
        # Store results
        lst_sigma_tot_born.append(sigma_tot_born_value)
        lst_sqrt_s.append(sqrt_s_val)
        lst_amp_born.append(amp_born_value)
    
    return lst_sigma_tot_born, lst_sqrt_s, lst_amp_born



In [6]:
mass_model = 'pl'
ensemble = 'atlas'
m2_func = get_m2_function(mass_model)
params = model_params[ensemble][mass_model]
mg, a1, a2 = params['mg'], params['a1'], params['a2']
epsilon = epsilon_values[ensemble]

# Calculate all cross sections
lst_sigma_tot_born, lst_sqrt_s, lst_amp_born = calculate_born_cross_sections(
    start_sqrt_s, max_sqrt_s, step, mg, a1, a2, m2_func, epsilon, n_points
)

lst_s = [val**2 for val in lst_sqrt_s]

In [11]:
import numpy as np
from scipy.integrate import fixed_quad
from scipy.special import j0

n = 2
q_max = 0.1
b_max = 10

# global lists
chi_list = []
amp_list = []

# -------------------------------
# Full chi(s,b) including k, phi, and new q integral
# -------------------------------
def chi_integral(sqrt_s_values, mg, a1, a2, m2_func, epsilon, b, q_max):
    global chi_list
    chi_list = []  # reset each time
    
    for sqrt_s_val in sqrt_s_values:
        s = sqrt_s_val**2

        def q_integrand(q):
            # Make sure q is iterable, even if scalar
            q = np.atleast_1d(q)
            out = []
            for qq in q:  # loop over each quadrature node
                t = -(qq**2)
                diff_T = compute_double_integral(sqrt_s_val, mg, a1, a2, m2_func, t)
                val = (qq * j0(b * qq) * born_amp(diff_T, s, epsilon, t)) / s
                out.append(val)
            return np.array(out)

        # integrate real and imaginary parts separately
        real_part, _ = fixed_quad(lambda q: np.real(q_integrand(q)), 0, q_max, n=n)
        imag_part, _ = fixed_quad(lambda q: np.imag(q_integrand(q)), 0, q_max, n=n)

        chi_list.append(real_part + 1j * imag_part)

    return chi_list


def eikonal_amplitude(sqrt_s_values, chi_list, b_max):
    global amp_list
    amp_list = []  # reset each time
    
    for sqrt_s_val, chi_val in zip(sqrt_s_values, chi_list):
        s = sqrt_s_val**2

        def b_integrand(b):
            b = np.atleast_1d(b)
            out = []
            for bb in b:  # loop over quadrature nodes in b
                val = bb * (1 - np.exp(1j * chi_val))
                out.append(val)
            return np.array(out)

        result, _ = fixed_quad(b_integrand, 0, b_max, n=n)
        A_eik = 1j * s * result
        amp_list.append(A_eik)

    return amp_list


# Example: loop over many b values
lst_b = np.linspace(0.1, b_max, 10)
lst_q = np.linspace(0, q_max, 10)
temp = []

for b_val in lst_b:
    for q_val in lst_q:
        chi_list = chi_integral(lst_sqrt_s, mg, a1, a2, m2_func, epsilon, b_val, q_val)
        A_eik_values = eikonal_amplitude(lst_sqrt_s, chi_list, b_val)
        temp.append(A_eik_values[0])
        print(f"b = {b_val:.2f}, q = {q_val:.2f}, A_eik[0] = {A_eik_values[0]}")

b = 0.10, q = 0.00, A_eik[0] = 0j
b = 0.10, q = 0.01, A_eik[0] = 0.4088045651241816j
b = 0.10, q = 0.02, A_eik[0] = 1.6149832334980898j
b = 0.10, q = 0.03, A_eik[0] = 3.559540874762685j
b = 0.10, q = 0.04, A_eik[0] = 6.149645676190795j
b = 0.10, q = 0.06, A_eik[0] = 9.266179334849857j
b = 0.10, q = 0.07, A_eik[0] = 12.773038452787398j
b = 0.10, q = 0.08, A_eik[0] = 16.527155253638334j
b = 0.10, q = 0.09, A_eik[0] = 20.38819829558502j
b = 0.10, q = 0.10, A_eik[0] = 24.22704512290938j
b = 1.20, q = 0.00, A_eik[0] = 0j
b = 1.20, q = 0.01, A_eik[0] = 58.866563557377944j
b = 1.20, q = 0.02, A_eik[0] = 232.53738894416463j
b = 1.20, q = 0.03, A_eik[0] = 512.4757542780271j
b = 1.20, q = 0.04, A_eik[0] = 885.2561320527286j
b = 1.20, q = 0.06, A_eik[0] = 1333.6656126528596j
b = 1.20, q = 0.07, A_eik[0] = 1838.0584073547216j
b = 1.20, q = 0.08, A_eik[0] = 2377.811524717759j
b = 1.20, q = 0.09, A_eik[0] = 2932.7293147401406j
b = 1.20, q = 0.10, A_eik[0] = 3484.264714137792j
b = 2.30, q = 0.00, A_e

In [13]:
sum = 0
temp_amp = []
for i in sorted(temp):
    sum += i 
    print(sum) 
    temp_amp.append(sum)

0j
0j
0j
0j
0j
0j
0j
0j
0j
0j
0.4088045651241816j
2.0237877986222714j
5.5833286733849565j
11.732974349575752j
20.999153684425607j
33.772192137213004j
50.29934739085134j
70.68754568643635j
94.91459080934573j
153.78115436672368j
370.0212201178007j
602.5586090619653j
1075.0527993062392j
1587.5285535842663j
2415.1003004575023j
3269.1524999903986j
4154.408632043127j
5435.802117916004j
6769.467730568864j
8603.325841832755j
10441.384249187477j
12306.995556373706j
14188.66180257781j
16566.473327295567j
19051.31562526032j
21984.044940000458j
25218.245652626625j
28467.436942717617j
31733.75942545935j
35218.02413959714j
39299.790177782306j
43408.25900453341j
48301.0611878474j
53356.0111423583j
60095.875424519225j
67185.88926139296j
74374.44129620578j
81604.35650632439j
90318.76200657511j
100108.06002755743j
110776.73057754026j
121519.08084509477j
132635.001870199j
145029.81221225037j
157760.64990119904j
170516.6663567868j
185200.73142572775j
201083.65714692773j
217135.59166142836j
235767.59831539

In [14]:
print(temp_amp)

[0j, 0j, 0j, 0j, 0j, 0j, 0j, 0j, 0j, 0j, 0.4088045651241816j, 2.0237877986222714j, 5.5833286733849565j, 11.732974349575752j, 20.999153684425607j, 33.772192137213004j, 50.29934739085134j, 70.68754568643635j, 94.91459080934573j, 153.78115436672368j, 370.0212201178007j, 602.5586090619653j, 1075.0527993062392j, 1587.5285535842663j, 2415.1003004575023j, 3269.1524999903986j, 4154.408632043127j, 5435.802117916004j, 6769.467730568864j, 8603.325841832755j, 10441.384249187477j, 12306.995556373706j, 14188.66180257781j, 16566.473327295567j, 19051.31562526032j, 21984.044940000458j, 25218.245652626625j, 28467.436942717617j, 31733.75942545935j, 35218.02413959714j, 39299.790177782306j, 43408.25900453341j, 48301.0611878474j, 53356.0111423583j, 60095.875424519225j, 67185.88926139296j, 74374.44129620578j, 81604.35650632439j, 90318.76200657511j, 100108.06002755743j, 110776.73057754026j, 121519.08084509477j, 132635.001870199j, 145029.81221225037j, 157760.64990119904j, 170516.6663567868j, 185200.73142572775

In [15]:
def sigma_tot_eik(amp, s):
    return (4*np.pi)/s * amp.imag * 0.389379323   

lst_sigma_tot_eik = [sigma_tot_eik(amp, s) for amp, s in zip(temp_amp, lst_s)]
print(lst_sigma_tot_eik)


[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.6501516146741551e-06, 6.8653345932209026e-06, 1.6140662285520162e-05, 2.924924096499593e-05, 4.5606123315599434e-05, 6.447024747245392e-05, 8.506217296256893e-05, 0.00010663462034056914, 0.00012851431205179484, 0.00018792808533957028, 0.00041016375065850693, 0.0006086140310748016, 0.000993526159960382, 0.0013474726995606114, 0.0018892548170040481, 0.002364488605130623, 0.002786395713231561, 0.0033901598063447347, 0.003935879688011657, 0.0046743059005784, 0.005312969167437884, 0.005877104937088365, 0.006371374575198724, 0.007008088078306636, 0.007605425215803652, 0.008295529463113635, 0.009008644527758994, 0.009641295142681194, 0.010203575306708105, 0.010764915713397159, 0.011433875073788152, 0.012035101418297897, 0.012776165738234492, 0.013479176718533414, 0.014514744312095325, 0.015529459947713656, 0.016467435218827403, 0.01732338200379067, 0.018398877200115685, 0.019585654359864216, 0.020831504941454488, 0.021981307316095166, 0.023

In [16]:
def add_iterative_curve(fig, x_data, y_data, 
                        curve_name:str=None, color:str='blue', line_type:str='lines+markers'):

    fig.add_trace(go.Scatter(
    x = x_data,
    y = y_data,
    mode=line_type, 
    name=curve_name,
    line=dict(
        color=color,
        width=2),
    marker=dict(size=4))
)
    fig.update_xaxes(gridcolor='lightgray')
    fig.update_yaxes(gridcolor='lightgray')

fig_amp = go.Figure()

add_iterative_curve(fig_amp, lst_sqrt_s, np.imag(lst_amp_born), curve_name='Im(A born)', color='blue')
add_iterative_curve(fig_amp, lst_sqrt_s, np.imag(temp_amp), curve_name='Im(A eikonal)', color='red')
fig_amp.update_layout(
    title='Amplitude Im vs. sqrt(s)',
    xaxis=dict(
        title='sqrt(s) [GeV]',
        type='log',
    ),
    yaxis=dict(
        title='Amp',
    ),
    showlegend=True,
    legend=dict(
        title='Model/Data'
    ),
    plot_bgcolor='white',
    hovermode='x unified'
)
fig_amp.show(renderer = 'browser')


fig_sigma = go.Figure()

add_iterative_curve(fig_sigma, lst_sqrt_s, lst_sigma_tot_born, curve_name='sigma tot born')
add_iterative_curve(fig_sigma, lst_sqrt_s, lst_sigma_tot_eik, curve_name='sigma tot eikonal', color='red')


# Add ATLAS data
fig_sigma.add_trace(go.Scatter(
    x=x_atlas,
    y=y_atlas,
    mode='markers',
    marker=dict(
        color='black',
        size=6,
        symbol='square'
    ),
    error_y=dict(
        type='data',
        array=y_error_atlas,
        visible=True
    ),
    name='ATLAS Data'
))

# Configure layout
fig_sigma.update_layout(
    title='Sigma Tot vs. sqrt(s)',
    xaxis=dict(
        title='sqrt(s) [GeV]',
        type='log',
    ),
    yaxis=dict(
        title='Sigma Tot [mb]',
    ),
    showlegend=True,
    legend=dict(
        title='Model/Data'
    ),
    plot_bgcolor='white',
    hovermode='x unified'
)

fig_sigma.update_xaxes(gridcolor='lightgray')
fig_sigma.update_yaxes(gridcolor='lightgray')

fig_sigma.show(renderer = 'browser')

Opening in existing browser session.
Opening in existing browser session.
